# LeadSignal-CS on free Google Colab

Created by Mazid. Run the cells in order. Cells 1 to 5 are setup and checks. Cells 6 onward are the real work.

- **No GPU needed.** Runtime type: CPU is fine.
- **Everything is saved to Google Drive**, so a disconnect never loses finished work. Rerun from the top and each stage continues from its last saved file.
- **If a cell stops with an error,** copy the error text and report it. `RUNBOOK.md` explains the common ones.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Get the code
Upload `LeadSignal-CS-0.1.0.zip` to `MyDrive/leadsignal/` once. This cell unzips it into the session.

In [ ]:
import os, sys, zipfile, shutil
DATA_DIR = '/content/drive/MyDrive/leadsignal'          # all results live here
ZIP = f'{DATA_DIR}/LeadSignal-CS-0.1.0.zip'
os.makedirs(DATA_DIR, exist_ok=True)
shutil.rmtree('/content/leadsignal-cs', ignore_errors=True)
zipfile.ZipFile(ZIP).extractall('/content/')
os.chdir('/content/leadsignal-cs'); sys.path.insert(0, '/content/leadsignal-cs')
print('ready:', os.path.exists('run_leadsignal.py'))

## 3. Settings
The email is a courtesy for OpenAlex, not a password. The GitHub token is optional (see `RUNBOOK.md`) and is read from Colab Secrets, never written to disk.

In [ ]:
CONTACT_EMAIL = 'your.email@example.com'   # change this
try:
    from google.colab import userdata
    os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
    print('GitHub token loaded from Colab Secrets')
except Exception as exc:  # the token is optional
    print('No GitHub token; GitHub snapshot will run slower:', type(exc).__name__)

## 4. Check the build (about one minute). Expect `28/28 checks passed`.

In [ ]:
!python tests/test_leadsignal.py

## 5. Synthetic dry run (about one minute)
This uses made-up data with planted signals to show every stage working. Its numbers are **not findings**, and the exporter will refuse to publish them.

In [ ]:
!python run_leadsignal.py demo --data-dir /content/demo_run

## 6. Check the data sources (under a minute)

In [ ]:
!python run_leadsignal.py diagnose --data-dir $DATA_DIR

## 6b. Storage check (run any time)
Shows how much of your Google Drive the project uses. Your rule: stay well under 10 GB. It warns at 8 GB. Google Drive also counts the **Trash**, so empty old LeadSignal files from it now and then.

In [ ]:
!python run_leadsignal.py storage --data-dir $DATA_DIR

## 7. First build of the arXiv corpus (only once, ever)
**If `stage1/arxiv_papers.parquet` is already in your Drive, skip this section and go to 8.**

**Option A (fastest):** run `kaggle_filter_snapshot.py` once in a free Kaggle Notebook with the dataset *Cornell-University/arxiv* attached. Download `cs_arxiv_2012plus.jsonl.gz`, put it in `MyDrive/leadsignal/`, then run the first cell below.

**Option B (no Kaggle):** harvest through OAI-PMH. Slower (about 4 hours) but resumable. If it stops, run section 8; it continues where it stopped.

In [ ]:
!python run_leadsignal.py build --data-dir $DATA_DIR --snapshot $DATA_DIR/cs_arxiv_2012plus.jsonl.gz

In [ ]:
# Option B (no Kaggle): download directly from arXiv. If it stops, run section 8 to continue.
!python run_leadsignal.py build --data-dir $DATA_DIR

## 8. Update the database (the "update" button)
Fetches only what is new since the last run and merges it. Safe to run any time and as often as you like.

In [ ]:
!python run_leadsignal.py update --data-dir $DATA_DIR

## 9. Validation snapshots (run by hand, as often as you like)
Each run saves a dated file. OpenAlex needs no key. GitHub works without a token, only slower.

In [ ]:
!python run_leadsignal.py openalex --data-dir $DATA_DIR --email $CONTACT_EMAIL

In [ ]:
!python run_leadsignal.py github --data-dir $DATA_DIR

## 10. Analyse: a quick sample first, then the full run

In [ ]:
!python run_leadsignal.py analyse --data-dir $DATA_DIR --sample

In [ ]:
!python run_leadsignal.py analyse --data-dir $DATA_DIR --events data/ground_truth/known_shift_events.csv

## 11. Precision audit
Writes one Excel sheet per concept to `MyDrive/leadsignal/audit/`. In Google Drive, double-click a sheet: it opens in Google Sheets and saves straight back to the same file. Type `1` (about the concept) or `0` (not about it) in the `relevant` column for each of the 30 papers. Then run the second cell. Existing sheets are never overwritten, so you can stop and continue another day.

In [ ]:
!python run_leadsignal.py audit --data-dir $DATA_DIR

In [ ]:
!python run_leadsignal.py precision --data-dir $DATA_DIR

## 12. Read the curriculum-review report

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f'{DATA_DIR}/stage6/curriculum_review_report.md').read()))

## 13. Export a citable bundle
Upload the folder this creates to Zenodo to get a DOI. The export refuses synthetic data.

In [ ]:
!python run_leadsignal.py export --data-dir $DATA_DIR

## 14. Curriculum events (after you check the mapping workbook)
Follow the **Mapping Checking Guide**. Cell 14a puts the workbook pre-filled from Claude's draft in `MyDrive/leadsignal/ground_truth/` (only if none is there; your file is never overwritten). Cell 14b reads it, shows how many rows you have checked, reports how often you agreed with the draft, and writes `curriculum_events.csv`. Run 14b as often as you like. Cell 14c runs the full analysis against the curriculum events (RQ2). Its results are saved under the name `curriculum_events`, so they never overwrite the known-shift check from Cell 10b.

In [ ]:
# 14a. Workbook pre-filled from Claude's draft, in Drive (skipped if one already exists)
!python tools/make_mapping_workbook.py $DATA_DIR/ground_truth/cs2013_cs2023_mapping.xlsx
# For a colleague acting as second coder, make an EMPTY copy instead (they must not see the draft):
# !python tools/make_mapping_workbook.py $DATA_DIR/ground_truth/second_coder.xlsx --blank

In [ ]:
# 14b. Read the filled workbook and write the curriculum events file
!python run_leadsignal.py events --data-dir $DATA_DIR
# Optional, once a colleague has coded some concepts in their own copy:
# !python run_leadsignal.py events --data-dir $DATA_DIR --second-coder $DATA_DIR/ground_truth/second_coder.xlsx

In [ ]:
# 14c. Full analysis against the curriculum events (only when 14b says 50 of 50)
!python run_leadsignal.py analyse --data-dir $DATA_DIR --events $DATA_DIR/ground_truth/curriculum_events.csv